## Imports and config

In [16]:
import ast
import csv
import os

QA_DIR_PATH = "../qa_datasets"
RESULTS_DIR_PATH = "../results"


In [18]:
eval_graphs = os.listdir(RESULTS_DIR_PATH)
for eval_graph in eval_graphs:
    print(eval_graph)

    eval_graph_path = os.path.join(RESULTS_DIR_PATH, eval_graph)
    eval_datasets = os.listdir(eval_graph_path)
    for eval_dataset in eval_datasets:
        qa_file_path = os.path.join(QA_DIR_PATH, f"{eval_dataset}.csv")
        if not os.path.exists(qa_file_path):
            print(f"{eval_dataset} does not exist in QA folder, skipping...")
            continue

        print(eval_dataset)
        eval_dataset_path = os.path.join(eval_graph_path, eval_dataset)

        retrievers_data = {}
        retriever_metrics = {}
        for retriever_file in os.listdir(eval_dataset_path):
            if not retriever_file.endswith(".csv"):
                continue

            retriever_name = os.path.splitext(retriever_file)[0]

            retrievers_data[retriever_name] = {}
            with open(
                os.path.join(eval_dataset_path, retriever_file),
                mode="r",
                encoding="utf-8",
            ) as f:
                reader = csv.reader(f)
                next(reader)

                for row in reader:
                    q_id = str(row[0]).strip()
                    retrievers_data[retriever_name][q_id] = ast.literal_eval(row[1])

            retriever_metrics[retriever_name] = {
                "Hit@1": [],
                "Hit@5": [],
                "Recall@20": [],
                "MRR": [],
            }

        with open(qa_file_path, mode="r", encoding="utf-8") as dataset_file:
            dataset_reader = csv.DictReader(dataset_file)

            for row in dataset_reader:
                query_id = row["id"]
                answer_ids = set(ast.literal_eval(row["answer_ids"]))

                for retriever_name, retr_ids_per_query in retrievers_data.items():
                    retrieved_ids = retr_ids_per_query[query_id]

                    # Hit@1
                    hit1 = (
                        1
                        if len(retrieved_ids) > 0
                        and retrieved_ids[0] in answer_ids
                        else 0
                    )

                    # Hit@5
                    hit5 = (
                        1
                        if any(retr_id in answer_ids for retr_id in retrieved_ids[:5])
                        else 0
                    )

                    # Recall@20
                    recall20 = len(set(retrieved_ids[:20]) & answer_ids) / len(answer_ids)

                    # MRR
                    mrr = 0.0
                    for rank, retr_id in enumerate(retrieved_ids, start=1):
                        if retr_id in answer_ids:
                            mrr = 1.0 / rank
                            break

                    retriever_metrics[retriever_name]["Hit@1"].append(hit1)
                    retriever_metrics[retriever_name]["Hit@5"].append(hit5)
                    retriever_metrics[retriever_name]["Recall@20"].append(
                        recall20
                    )
                    retriever_metrics[retriever_name]["MRR"].append(mrr)

        for retriever_name, metrics in retriever_metrics.items():
            for key, values in metrics.items():
                avg_value = sum(values) / len(values) if values else 0.0
                print(f"  {retriever_name} - {key}: {round(avg_value, 4)}")


stark-amazon
text-amazon
  VSSRetriever - Hit@1: 0.48
  VSSRetriever - Hit@5: 0.6545
  VSSRetriever - Recall@20: 0.7709
  VSSRetriever - MRR: 0.5614
  StructuralRetriever - Hit@1: 0.0109
  StructuralRetriever - Hit@5: 0.0836
  StructuralRetriever - Recall@20: 0.12
  StructuralRetriever - MRR: 0.0407
  AgenticRetriever - Hit@1: 0.6182
  AgenticRetriever - Hit@5: 0.6327
  AgenticRetriever - Recall@20: 0.6327
  AgenticRetriever - MRR: 0.6255
rel-amazon
  AgenticRetriever - Hit@1: 0.7191
  AgenticRetriever - Hit@5: 0.7759
  AgenticRetriever - Recall@20: 0.7625
  AgenticRetriever - MRR: 0.7424
  StructuralRetriever - Hit@1: 0.291
  StructuralRetriever - Hit@5: 0.5753
  StructuralRetriever - Recall@20: 0.6102
  StructuralRetriever - MRR: 0.4207
  VSSRetriever - Hit@1: 0.0502
  VSSRetriever - Hit@5: 0.1204
  VSSRetriever - Recall@20: 0.11
  VSSRetriever - MRR: 0.084
stark-amazon
  StructuralRetriever - Hit@1: 0.1396
  StructuralRetriever - Hit@5: 0.4297
  StructuralRetriever - Recall@20: 0.30